# Imports

In [1]:
from pathlib import Path

from sentence_transformers import CrossEncoder
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_core.prompts import ChatPromptTemplate
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_community.chat_models import ChatOllama
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_experimental.text_splitter import SemanticChunker

## Functions

In [2]:
def load_embedding():
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2")
    return embeddings

def load_vector_store():
    embeddings = load_embedding()
    db = FAISS.load_local(
        "../vector_store",
        embeddings,
        allow_dangerous_deserialization=True
    )
    return db

def vector_search(query, k=5):
    db = load_vector_store()
    docs = db.similarity_search(query, k=k)
    return docs

def get_bm25_retriver(chunks, k=10):
    bm25_retriever = BM25Retriever.from_documents(chunks)
    bm25_retriever.k = k
    return bm25_retriever

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query, docs, top_k=5):

    pairs = [(query, doc.page_content) for doc in docs]
    
    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(scores, docs),
        key=lambda x: x[0],
        reverse=True
    )

    return [doc for _, doc in ranked[:top_k]]


def hybrid_search(query, k=10):

    docs_bm25 = bm25_retriever.invoke(query)
    docs_vector = vector_retriever.invoke(query)

    docs = docs_bm25 + docs_vector

    unique_docs = {doc.page_content: doc for doc in docs}

    return list(unique_docs.values())[:k]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Model

In [3]:
def invoke(query):
    
    docs = hybrid_search(query)

    reranked_docs = rerank(query, docs)
    
    context = "\n\n".join(doc.page_content for doc in reranked_docs)
    
    prompt = get_prompt()

    return llm.invoke(prompt.format(context=context, question=query))

In [4]:
llm = ChatOllama(model="qwen2.5:7b-instruct")

/tmp/ipykernel_10743/1007991676.py:1: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  llm = ChatOllama(model="qwen2.5:7b-instruct")


In [5]:
prompt = ChatPromptTemplate.from_template(
    """
    Você é um assistente que responde perguntas usando apenas o contexo apresentado.
    
    Contexto:
    {context}
    
    Pergunta:
    {question}
    
    Resposta:
    """
)

In [6]:
resposta = invoke("quando um pedido é indeferido?")

NameError: name 'bm25_retriever' is not defined

In [18]:
print(resposta.content)

Um pedido é indeferido quando a autoridade administrativa, após analisar o mérito do caso, os documentos apresentados e a legislação aplicável, conclui que não há fundamento para sua aprovação. No contexto fornecido, isso ocorre nas seguintes decisões:

1. **Processo nº 54220/2022**: A decisão é de indeferimento, pois foram analisados os argumentos apresentados e documentos anexados, mas não foram identificados vícios formais ou materiais que justificassem a afastamento da penalidade já aplicada.

2. **Processo nº 60353/2022**: A decisão é também de indeferimento, pois após análise dos documentos e registros constantes no sistema, não foram encontrados elementos novos ou relevantes que justificassem a alteração da decisão anterior.

Portanto, um pedido é indeferido quando os elementos apresentados não convencem a autoridade competente, mantendo-se assim a decisão original ou aplicando penalidades conforme já estabelecidas.


In [19]:
resposta = invoke("Me dê um motivo para deferimento?")

In [20]:
resposta

AIMessage(content='Um motivo para o deferimento neste caso é a comprovação de que há inconsistências materiais no auto de infração, que podem comprometer a validade do ato administrativo original. Diante disso, acolheu-se o recurso apresentado pelo interessado, resultando no cancelamento integral da penalidade aplicada.', additional_kwargs={}, response_metadata={'model': 'qwen2.5:7b-instruct', 'created_at': '2026-03-13T17:10:47.569732835Z', 'message': {'role': 'assistant', 'content': ''}, 'done': True, 'done_reason': 'stop', 'total_duration': 33595838234, 'load_duration': 80482755, 'prompt_eval_count': 868, 'prompt_eval_duration': 24345321617, 'eval_count': 73, 'eval_duration': 9120199196}, id='lc_run--019ce82d-60d4-7213-9cce-f63f791c7397-0', tool_calls=[], invalid_tool_calls=[])

In [ ]:
# O GPT indicou salvar os documentos com metadados